#KITCHEN PROLINE - Notebook 03 - Expot CSV

**Objectif :**
Ce notebook exporte l'ensemble des tables du modèle décisionnel (dimensions et table de faits) au format CSV afin de les connecter à Microsoft Power BI pour la visualisation et le reporting.

**Tables exportées :**
Dimensions : DimPays, DimMagasin, DimClient, DimVendeur, DimProduit, DimFamilleProduit, DimMarque, DimFournisseur, DimModele, DimFacade, DimDevis
Table de faits : FactLignesDevis

**Destination :**
`/Volumes/kitchen-proline-data/raw/exports/csv/`
Convention de nommage : DimXxx.csv, FactLignesDevis.csv

**Format de sortie :**
Séparateur virgule, encodage UTF-8, en-têtes inclus, un seul fichier par table (coalesce(1))

**Dépendances :**
Ce notebook doit être exécuté APRÈS les notebooks 01 et 02


In [0]:
from pyspark.sql import functions as F

# Configuration
CATALOG = "kitchen-proline-data"
SCHEMA = "sales"
EXPORT_PATH = "/Volumes/kitchen-proline-data/raw/exports/csv/"

# Liste des tables à exporter
TABLES = [
    # Dimensions de référence
    "DimPays",
    "DimMagasin",
    "DimClient",
    "DimVendeur",
    "DimProduit",
    "DimFamilleProduit",
    "DimMarque",
    "DimFournisseur",
    "DimModele",
    "DimFacade",
    # Dimension temporelle
    "DimDevis",
    # Table de faits
    "FactLignesDevis"
]

print(f" Configuration chargée")
print(f" Répertoire d'export : {EXPORT_PATH}")
print(f"Nombre de tables à exporter : {len(TABLES)}")

In [0]:
def export_table_to_csv(table_name, export_path):
    """
    Exporte une table Unity Catalog vers un fichier CSV unique
    """
    print(f"\n Export de {table_name}...")
    
    # Lire la table
    full_table_name = f"`{CATALOG}`.`{SCHEMA}`.`{table_name}`"
    df = spark.table(full_table_name)
    
    row_count = df.count()
    print(f"Lignes : {row_count}")
    print(f"Colonnes : {len(df.columns)}")
    
    # Créer le chemin de destinaton
    output_path = f"{export_path}{table_name}"
    
    # Export en CSV avec unseul fichier (coalesce(1))
    df.coalesce(1).write.mode("overwrite").option("header", "true").option("sep", ",").csv(output_path)
    
    print(f" Exporté vers {output_path}")
    return row_count

print(" Fonction d'export prête")

## Export de tous les tables

In [0]:
print("="*70)
print("EXPORT DES TABLES EN CSV")
print("="*70)

# Dictionnaire pour stocker les statisiques
stats = {}

# Exporter chaque table
for table_name in TABLES:
    try:
        row_count = export_table_to_csv(table_name, EXPORT_PATH)
        stats[table_name] = {"status": "", "rows": row_count}
    except Exception as e:
        print(f"Erreur : {str(e)}")
        stats[table_name] = {"status": "", "rows": 0}

print("\n" + "="*70)
print("RÉCAPITULATIF DES EXPORTS")
print("="*70)

# Aficher le récapitulatif
for table_name, stat in stats.items():
    if stat["status"] == "":
        print(f"{stat['status']} {table_name:25s} : {stat['rows']:>8,} lignes")
    else:
        print(f"{stat['status']} {table_name:25s} : Échec")

total_rows = sum(s["rows"] for s in stats.values())
print(f"\n Total : {total_rows:,} lignes exportées")
print(f" Tous les exports sont dans : {EXPORT_PATH}")

##  Vérification des fichiers exportés

In [0]:
print("Fichiers CSV créés :\n")

# Lister les répertoires créés
try:
    directories = dbutils.fs.ls(EXPORT_PATH)
    
    for directory in sorted(directories, key=lambda x: x.name):
        # Chaque export crée un répertoire, lister les fichiers à l'intérieur
        table_name = directory.name.rstrip('/')
        files = dbutils.fs.ls(directory.path)
        
        # Trouver le fichier CSV (ignorer _SUCCESS et autres métadonées
        csv_files = [f for f in files if f.name.endswith('.csv')]
        
        if csv_files:
            csv_file = csv_files[0]
            size_mb = csv_file.size / (1024 * 1024)
            print(f"{table_name:25s} : {csv_file.name:40s} ({size_mb:.2f} MB)")
        else:
            print(f"  {table_name:25s} : (fichier CSV non touvé)")
            
    print(f"\n✓ Tous les exports sont disponibles dans : {EXPORT_PATH}")
except Exception as e:
    print(f" Erreur lors de la lecture : {str(e)}")

In [0]:
#Vérif rapide de la table des facts
fact = spark.table("`kitchen-proline-data`.`sales`.`FactLignesDevis`")
print("Nombre de colonnes :", len(fact.columns))
print("\nListe des colonnes :")
for col in fact.columns:
    print(f"  - {col}")